# Noise Review Gradio UI

从 `crops` 表读取 noise score，按分数分 bin 抽样，人工三分类后回写数据库。


In [1]:
from __future__ import annotations

import os
import re
import sqlite3
import sys
from pathlib import Path
from typing import Any

import gradio as gr
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"Project root not found from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import utils

DB_PATH = Path(utils.resolve_db_path())
utils.init_db(DB_PATH)
DATA_ROOT = DB_PATH.parent
DB_PATH, DATA_ROOT


(PosixPath('/Users/yukun/projects/wakareeru/data/commons_image_manifest.sqlite'),
 PosixPath('/Users/yukun/projects/wakareeru/data'))

In [2]:
REVIEW_CONFIG = {
    "score_col": "noise_score_v1",
    "skip_reviewed": True,
    "sampling_density": "quantile",  # quantile | linear | high_dense
    "bin_count": 5,
    "samples_per_bin": 8,
    "random_seed": 42,
    "pad_frac": 0.04,
    "review_labels": ["ok", "ambiguous", "bad"],
    "review_label_col": "noise_review_label",
    "review_note_col": "noise_review_note",
    "reviewed_at_col": "noise_reviewed_at",
    "review_score_col": "noise_review_score_col",
}


In [3]:
REVIEW_COLUMN_DEFS = {
    REVIEW_CONFIG["review_label_col"]: "TEXT",
    REVIEW_CONFIG["review_note_col"]: "TEXT",
    REVIEW_CONFIG["reviewed_at_col"]: "TEXT",
    REVIEW_CONFIG["review_score_col"]: "TEXT",
}


def quote_ident(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Unsafe SQLite identifier: {name!r}")
    return f'"{name}"'


def crop_columns(conn: sqlite3.Connection) -> dict[str, str]:
    rows = conn.execute("PRAGMA table_info(crops)").fetchall()
    return {row[1]: (row[2] or "") for row in rows}


def ensure_review_columns(db_path: Path = DB_PATH) -> None:
    with sqlite3.connect(db_path) as conn:
        cols = crop_columns(conn)
        for col, col_type in REVIEW_COLUMN_DEFS.items():
            if col not in cols:
                conn.execute(f"ALTER TABLE crops ADD COLUMN {quote_ident(col)} {col_type}")
        conn.commit()


def score_columns(db_path: Path = DB_PATH) -> list[str]:
    with sqlite3.connect(db_path) as conn:
        cols = crop_columns(conn)
    preferred = [c for c in cols if c.startswith("noise_score")]
    numeric = [
        c for c, t in cols.items()
        if c not in preferred and any(token in t.upper() for token in ["REAL", "INT", "NUM"])
    ]
    return preferred + numeric


ensure_review_columns()
score_columns()


['noise_score_v1',
 'id',
 'image_id',
 'crop_index',
 'source_result_index',
 'detector_score',
 'box_x1',
 'box_y1',
 'box_x2',
 'box_y2',
 'box_area',
 'nms_iou_threshold']

In [4]:
def load_candidate_rows(
    score_col: str,
    skip_reviewed: bool = True,
    db_path: Path = DB_PATH,
) -> pd.DataFrame:
    ensure_review_columns(db_path)
    with sqlite3.connect(db_path) as conn:
        cols = crop_columns(conn)
        if score_col not in cols:
            raise ValueError(f"score_col={score_col!r} not found in crops table")

        score_expr = f"c.{quote_ident(score_col)}"
        review_label_col = quote_ident(REVIEW_CONFIG["review_label_col"])
        review_note_col = quote_ident(REVIEW_CONFIG["review_note_col"])
        reviewed_at_col = quote_ident(REVIEW_CONFIG["reviewed_at_col"])
        where = [f"{score_expr} IS NOT NULL", "i.downloaded_path IS NOT NULL"]
        if skip_reviewed:
            where.append(f"(c.{review_label_col} IS NULL OR c.{review_label_col} = '')")

        sql = f"""
            SELECT
                c.id AS crop_id,
                c.image_id,
                c.series,
                c.power_type,
                c.crop_index,
                c.detector_label,
                c.detector_score,
                c.box_x1,
                c.box_y1,
                c.box_x2,
                c.box_y2,
                c.box_area,
                c.crop_status,
                c.crop_reason,
                {score_expr} AS score,
                c.{review_label_col} AS review_label,
                c.{review_note_col} AS review_note,
                c.{reviewed_at_col} AS reviewed_at,
                i.file_title,
                i.downloaded_path,
                COALESCE(i.fine_grained_series, i.series) AS image_label,
                i.category
            FROM crops c
            JOIN images i ON i.id = c.image_id
            WHERE {' AND '.join(where)}
            ORDER BY score DESC
        """
        return pd.read_sql_query(sql, conn)


def assign_score_bins(df: pd.DataFrame, bins: int, density: str) -> pd.Series:
    if df.empty:
        return pd.Series(dtype="Int64")

    score = df["score"].astype(float)
    bins = max(1, int(bins))

    if density == "linear":
        return pd.cut(score, bins=bins, labels=False, duplicates="drop")

    rank_pct = score.rank(method="first", pct=True)
    if density == "high_dense":
        # More and narrower bins near high scores, where noisy samples usually concentrate.
        q = 1.0 - (1.0 - np.linspace(0, 1, bins + 1)) ** 2
        q[0], q[-1] = 0.0, 1.0
        edges = np.unique(q)
        return pd.cut(rank_pct, bins=edges, labels=False, include_lowest=True, duplicates="drop")

    if density == "quantile":
        return pd.qcut(score.rank(method="first"), q=bins, labels=False, duplicates="drop")

    raise ValueError("sampling_density must be one of: quantile, linear, high_dense")


def stratified_sample_rows(
    df: pd.DataFrame,
    bins: int,
    samples_per_bin: int,
    density: str,
    seed: int,
) -> pd.DataFrame:
    work = df.dropna(subset=["score"]).copy()
    if work.empty:
        return work.assign(score_bin=pd.Series(dtype="Int64"))

    work["score_bin"] = assign_score_bins(work, bins=bins, density=density)
    work = work.dropna(subset=["score_bin"]).copy()
    work["score_bin"] = work["score_bin"].astype(int)

    parts = []
    for _, group in work.groupby("score_bin", sort=False):
        n = min(int(samples_per_bin), len(group))
        parts.append(group.sample(n=n, random_state=int(seed)))
    if not parts:
        return work.head(0)

    sampled = pd.concat(parts, ignore_index=True)
    sampled = sampled.sort_values(["score_bin", "score"], ascending=[False, False]).reset_index(drop=True)
    return sampled


def load_review_sample(
    score_col: str,
    skip_reviewed: bool,
    density: str,
    bins: int,
    samples_per_bin: int,
    seed: int,
) -> pd.DataFrame:
    candidates = load_candidate_rows(score_col=score_col, skip_reviewed=skip_reviewed)
    return stratified_sample_rows(
        candidates,
        bins=bins,
        samples_per_bin=samples_per_bin,
        density=density,
        seed=seed,
    )


In [5]:
def placeholder_image(message: str, size: tuple[int, int] = (512, 384)) -> Image.Image:
    img = Image.new("RGB", size, "#f2f2f2")
    draw = ImageDraw.Draw(img)
    draw.multiline_text((20, 20), message, fill="#333333")
    return img


def load_crop_image(row: dict[str, Any], pad_frac: float | None = None) -> Image.Image:
    pad = REVIEW_CONFIG["pad_frac"] if pad_frac is None else float(pad_frac)
    try:
        return utils.load_crop(row, img_root=DATA_ROOT, pad_frac=pad)
    except Exception as exc:
        return placeholder_image(f"Failed to load crop_id={row.get('crop_id')}\n{exc}")


def row_markdown(row: dict[str, Any], idx: int, total: int) -> str:
    if not row:
        return "No sample loaded."

    fields = [
        ("progress", f"{idx + 1}/{total}"),
        ("crop_id", row.get("crop_id")),
        ("score", f"{float(row.get('score')):.6f}" if pd.notna(row.get("score")) else None),
        ("score_bin", row.get("score_bin")),
        ("series", row.get("series")),
        ("image_label", row.get("image_label")),
        ("power_type", row.get("power_type")),
        ("detector", row.get("detector_label")),
        ("detector_score", f"{float(row.get('detector_score')):.4f}" if pd.notna(row.get("detector_score")) else None),
        ("file_title", row.get("file_title")),
        ("category", row.get("category")),
        ("existing_review", row.get("review_label")),
        ("reviewed_at", row.get("reviewed_at")),
    ]
    lines = [f"### Crop review"]
    for key, value in fields:
        if value is not None and value == value:
            lines.append(f"- **{key}**: {value}")
    return "\n".join(lines)


def display_record(records: list[dict[str, Any]], idx: int):
    total = len(records or [])
    if total == 0:
        return placeholder_image("No rows sampled."), "No rows sampled.", None, "", "0/0"

    idx = max(0, min(int(idx), total - 1))
    row = records[idx]
    image = load_crop_image(row)
    md = row_markdown(row, idx, total)
    label = row.get("review_label") if row.get("review_label") else None
    note = row.get("review_note") or ""
    progress = f"{idx + 1}/{total}"
    return image, md, label, note, progress


def save_review(crop_id: int, label: str, note: str, score_col: str, db_path: Path = DB_PATH) -> None:
    if not label:
        raise gr.Error("请选择一个 review label 再保存。")

    ensure_review_columns(db_path)
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            f"""
            UPDATE crops
            SET {quote_ident(REVIEW_CONFIG['review_label_col'])} = ?,
                {quote_ident(REVIEW_CONFIG['review_note_col'])} = ?,
                {quote_ident(REVIEW_CONFIG['reviewed_at_col'])} = CURRENT_TIMESTAMP,
                {quote_ident(REVIEW_CONFIG['review_score_col'])} = ?
            WHERE id = ?
            """,
            (label, note or None, score_col, int(crop_id)),
        )
        conn.commit()


In [6]:
def build_review_app():
    score_choices = score_columns()
    default_score_col = REVIEW_CONFIG["score_col"] if REVIEW_CONFIG["score_col"] in score_choices else (score_choices[0] if score_choices else None)

    with gr.Blocks(title="Noise Crop Review") as app:
        gr.Markdown("# Noise Crop Review")

        records_state = gr.State([])
        idx_state = gr.State(0)

        with gr.Row():
            with gr.Column(scale=1):
                score_col = gr.Dropdown(
                    choices=score_choices,
                    value=default_score_col,
                    label="Noise score source (DB column)",
                    allow_custom_value=True,
                )
                skip_reviewed = gr.Checkbox(
                    value=REVIEW_CONFIG["skip_reviewed"],
                    label="Skip already reviewed crops",
                )
                density = gr.Radio(
                    choices=["high_dense", "quantile", "linear"],
                    value=REVIEW_CONFIG["sampling_density"],
                    label="Sampling score density",
                )
                bins = gr.Slider(2, 20, value=REVIEW_CONFIG["bin_count"], step=1, label="Bin count")
                samples_per_bin = gr.Slider(1, 50, value=REVIEW_CONFIG["samples_per_bin"], step=1, label="Samples per bin")
                seed = gr.Number(value=REVIEW_CONFIG["random_seed"], precision=0, label="Random seed")
                reload_btn = gr.Button("Load / resample", variant="primary")
                progress = gr.Textbox(label="Progress", interactive=False)

            with gr.Column(scale=2):
                image = gr.Image(label="Crop", type="pil", height=520)
                meta = gr.Markdown()

            with gr.Column(scale=1):
                review_label = gr.Radio(
                    choices=REVIEW_CONFIG["review_labels"],
                    label="Manual review label",
                )
                note = gr.Textbox(label="Note", lines=4, placeholder="optional")
                with gr.Row():
                    prev_btn = gr.Button("Previous")
                    skip_btn = gr.Button("Skip")
                save_next_btn = gr.Button("Save & next", variant="primary")

        sample_table = gr.Dataframe(label="Current sampled rows", interactive=False, wrap=True)

        def on_reload(score_col_value, skip_value, density_value, bins_value, samples_per_bin_value, seed_value):
            sample = load_review_sample(
                score_col=score_col_value,
                skip_reviewed=bool(skip_value),
                density=density_value,
                bins=int(bins_value),
                samples_per_bin=int(samples_per_bin_value),
                seed=int(seed_value),
            )
            records = sample.to_dict(orient="records")
            img, md, label, note_value, prog = display_record(records, 0)
            preview_cols = [
                c for c in [
                    "crop_id", "score", "score_bin", "series", "image_label",
                    "detector_label", "file_title", "review_label", "reviewed_at",
                ]
                if c in sample.columns
            ]
            return records, 0, img, md, label, note_value, prog, sample[preview_cols]

        def on_save_next(records, idx, label, note_value, score_col_value):
            records = records or []
            if not records:
                raise gr.Error("当前没有 sample，请先 Load / resample。")
            idx = max(0, min(int(idx), len(records) - 1))
            row = records[idx]
            save_review(row["crop_id"], label, note_value, score_col_value)
            row["review_label"] = label
            row["review_note"] = note_value
            idx = min(idx + 1, len(records) - 1)
            img, md, next_label, next_note, prog = display_record(records, idx)
            return records, idx, img, md, next_label, next_note, prog

        def on_skip(records, idx):
            records = records or []
            idx = min(int(idx) + 1, max(0, len(records) - 1))
            img, md, label, note_value, prog = display_record(records, idx)
            return idx, img, md, label, note_value, prog

        def on_prev(records, idx):
            records = records or []
            idx = max(0, int(idx) - 1)
            img, md, label, note_value, prog = display_record(records, idx)
            return idx, img, md, label, note_value, prog

        reload_btn.click(
            on_reload,
            inputs=[score_col, skip_reviewed, density, bins, samples_per_bin, seed],
            outputs=[records_state, idx_state, image, meta, review_label, note, progress, sample_table],
        )
        save_next_btn.click(
            on_save_next,
            inputs=[records_state, idx_state, review_label, note, score_col],
            outputs=[records_state, idx_state, image, meta, review_label, note, progress],
        )
        skip_btn.click(
            on_skip,
            inputs=[records_state, idx_state],
            outputs=[idx_state, image, meta, review_label, note, progress],
        )
        prev_btn.click(
            on_prev,
            inputs=[records_state, idx_state],
            outputs=[idx_state, image, meta, review_label, note, progress],
        )

    return app


In [7]:
app = build_review_app()
app.launch(server_name="127.0.0.1", server_port=7860, inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/anyio/to_thread.py", line 6